# Sentiment Analysis with TF-IDF and Logistic Regression

A complete end-to-end pipeline that classifies movie reviews as **positive** or **negative**.

**Pipeline:** Corpus → Text Cleaning → TF-IDF Features → Logistic Regression → Prediction

## Step 1: Create a Simple Corpus

A small, balanced dataset: 5 positive reviews and 5 negative reviews.

In [1]:
# The reviews (our text data / features)
reviews = [
    "This movie is amazing and fantastic",
    "I loved the acting and story",
    "Excellent film with great performance",
    "Very good movie and enjoyable",
    "Wonderful experience and brilliant screenplay",

    "This movie is terrible",
    "I hated the acting and story",
    "Very bad film and boring",
    "Worst movie I have ever watched",
    "Poor performance and disappointing experience"
]

# The sentiment labels (our target / what we want to predict)
sentiments = [
    "positive",
    "positive",
    "positive",
    "positive",
    "positive",

    "negative",
    "negative",
    "negative",
    "negative",
    "negative"
]

## Step 2: Import Libraries

In [2]:
import pandas as pd          # for building and viewing the DataFrame
import nltk                  # Natural Language Toolkit for text processing
import re                    # regular expressions for cleaning text

from nltk.corpus import stopwords        # list of common words to remove
from nltk.tokenize import word_tokenize  # splits text into words (tokens)

from sklearn.feature_extraction.text import TfidfVectorizer  # converts text -> numbers
from sklearn.model_selection import train_test_split         # splits data into train/test
from sklearn.linear_model import LogisticRegression          # the classification model
from sklearn.metrics import accuracy_score                   # to measure performance

## Step 3: Download NLTK Resources

Run only once. Downloads the tokenizer and the stopwords list.

> **Note:** Newer NLTK versions also need `punkt_tab`. It is included below so tokenization works reliably.

In [3]:
nltk.download('punkt')       # tokenizer models
nltk.download('punkt_tab')   # required by newer NLTK versions for word_tokenize
nltk.download('stopwords')   # common stopwords (the, is, and, ...)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Laptop\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Laptop\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Laptop\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Step 4: Create DataFrame

Put the reviews and sentiments together in a pandas DataFrame for easy handling.

In [5]:
# Build a DataFrame with two columns: review text and its sentiment
df = pd.DataFrame({
    "review": reviews,
    "sentiment": sentiments
})

# Show the first 5 rows
print(df.head(20))

                                          review sentiment
0            This movie is amazing and fantastic  positive
1                   I loved the acting and story  positive
2          Excellent film with great performance  positive
3                  Very good movie and enjoyable  positive
4  Wonderful experience and brilliant screenplay  positive
5                         This movie is terrible  negative
6                   I hated the acting and story  negative
7                       Very bad film and boring  negative
8                Worst movie I have ever watched  negative
9  Poor performance and disappointing experience  negative


## Step 5: Text Cleaning Function

Tasks performed on each review:
1. **Lowercase** the text
2. **Remove special characters** (keep only letters and spaces)
3. **Tokenize** (split into words)
4. **Remove stopwords** (drop common words like *the, is, and*)

In [7]:
# Load English stopwords once into a set (fast lookups)
stop_words = set(stopwords.words('english'))

def preprocess(text):
    # 1. lowercase so "Movie" and "movie" are treated the same
    text = text.lower()

    # 2. remove punctuation/special characters -> keep only letters and spaces
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 3. tokenize: break the sentence into individual words
    words = word_tokenize(text)

    # 4. remove stopwords: keep only meaningful words
    words = [word for word in words if word not in stop_words]

    # rejoin the cleaned words back into a single string
    return " ".join(words)

In [8]:
# Apply the cleaning function to every review and store the result in a new column
df["cleaned_review"] = df["review"].apply(preprocess)

# Compare original vs cleaned text
print(df[["review", "cleaned_review"]])

                                          review  \
0            This movie is amazing and fantastic   
1                   I loved the acting and story   
2          Excellent film with great performance   
3                  Very good movie and enjoyable   
4  Wonderful experience and brilliant screenplay   
5                         This movie is terrible   
6                   I hated the acting and story   
7                       Very bad film and boring   
8                Worst movie I have ever watched   
9  Poor performance and disappointing experience   

                              cleaned_review  
0                    movie amazing fantastic  
1                         loved acting story  
2           excellent film great performance  
3                       good movie enjoyable  
4  wonderful experience brilliant screenplay  
5                             movie terrible  
6                         hated acting story  
7                            bad film boring  
8   

**Example**

```
Original:  This movie is amazing and fantastic
Cleaned:   movie amazing fantastic
```

## Step 6: TF-IDF Feature Extraction

**What TF-IDF does:** converts words into numbers so the model can learn from them.

- **TF (Term Frequency):** how often a word appears in a review.
- **IDF (Inverse Document Frequency):** rare, informative words get higher weight; common words get lower weight.

In [ ]:
# Create the TF-IDF vectorizer
vectorizer = TfidfVectorizer()

# Learn the vocabulary AND transform the cleaned reviews into a numeric matrix
X = vectorizer.fit_transform(df["cleaned_review"])

# View the vocabulary (all unique words the model learned)
print(vectorizer.get_feature_names_out())

['acting' 'amazing' 'bad' 'boring' 'brilliant' 'disappointing' 'enjoyable'
 'ever' 'excellent' 'experience' 'fantastic' 'film' 'good' 'great' 'hated'
 'loved' 'movie' 'performance' 'poor' 'screenplay' 'story' 'terrible'
 'watched' 'wonderful' 'worst']


In [12]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 33 stored elements and shape (10, 25)>
  Coords	Values
  (0, 16)	0.4235494140918555
  (0, 1)	0.6405489418547368
  (0, 10)	0.6405489418547368
  (1, 15)	0.6394888567109641
  (1, 0)	0.5436239518925301
  (1, 20)	0.5436239518925301
  (2, 8)	0.5387481691380603
  (2, 11)	0.4579851637885095
  (2, 13)	0.5387481691380603
  (2, 17)	0.4579851637885095
  (3, 16)	0.4235494140918555
  (3, 12)	0.6405489418547368
  (3, 6)	0.6405489418547368
  (4, 23)	0.5182909034319405
  (4, 9)	0.44059461896295216
  (4, 4)	0.5182909034319405
  (4, 19)	0.5182909034319405
  (5, 16)	0.5515559913995145
  (5, 21)	0.8341378713086336
  (6, 0)	0.5436239518925301
  (6, 20)	0.5436239518925301
  (6, 14)	0.6394888567109641
  (7, 11)	0.5151921890284284
  (7, 2)	0.6060433187339399
  (7, 3)	0.6060433187339399
  (8, 16)	0.3566546401133593
  (8, 24)	0.5393815803571129
  (8, 7)	0.5393815803571129
  (8, 22)	0.5393815803571129
  (9, 17)	0.4579851637885095
  (9, 9)	0.45798516378

## Step 7: Encode Labels

Convert the text labels into numbers: **positive → 1**, **negative → 0**.

In [13]:
# Map the sentiment strings to numeric labels
y = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

print(y)

0    1
1    1
2    1
3    1
4    1
5    0
6    0
7    0
8    0
9    0
Name: sentiment, dtype: int64


## Step 8: Train-Test Split

Split the data so we train on part of it and test on data the model has never seen.

> **Note:** With only 10 rows, `test_size=0.2` gives just 2 test samples. `random_state=42` keeps the split reproducible.

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,                 # TF-IDF features
    y,                 # numeric labels
    test_size=0.2,     # 20% of the data used for testing
    random_state=42    # fixed seed -> same split every run
)

print("Training samples:", X_train.shape[0])
print("Testing samples: ", X_test.shape[0])

Training samples: 8
Testing samples:  2


## Step 9: Train Logistic Regression

The model reads the TF-IDF numbers and learns which words point to *positive* vs *negative*.

```
TF-IDF numbers
      ↓
Logistic Regression learns

amazing, fantastic, excellent  → positive
terrible, bad, worst           → negative
```

In [15]:
# Create the model
model = LogisticRegression()

# Train (fit) the model on the training data
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


## Step 10: Prediction & Accuracy

Use the trained model to predict the test set and measure accuracy.

In [16]:
# Predict sentiments for the unseen test data
predictions = model.predict(X_test)
print("Predictions:", predictions)

# Compare predictions with the true labels to get accuracy
accuracy = accuracy_score(y_test, predictions)
print("Accuracy:", accuracy)

Predictions: [1 0]
Accuracy: 0.0


## Step 11: Predict a New User Review

Reuse the **same** cleaning and vectorizer so the new review is processed exactly like the training data.

In [18]:
# Helper: clean -> vectorize -> predict -> return readable label
def predict_sentiment(new_review):
    clean_review = preprocess(new_review)                  # same cleaning as training
    vector_review = vectorizer.transform([clean_review])   # same TF-IDF vocabulary
    prediction = model.predict(vector_review)[0]           # 1 or 0
    return "positive" if prediction == 1 else "negative"

# Example 1 (expected: positive)
new_review = "This movie is fantastic and wonderful"
print(new_review, "->", predict_sentiment(new_review))

# Example 2 (expected: negative)
new_review = "This movie is boring and terrible"
print(new_review, "->", predict_sentiment(new_review))

This movie is fantastic and wonderful -> positive
This movie is boring and terrible -> negative
